# Can we predict `passed_quiz`?
Target: whether a student passes the quiz closing a tutoring session. Methodology follows `checklist.md`.

**Design decisions (made before any modelling)**
- `data.csv` has no session date, so there is no "later rows" ordering. The only repeated entity is `student_key` (1800 students, ~7 sessions each), so the test set and all CV folds hold **whole students** (grouped splits).
- Identifier `student_key` is dropped from the features (used only for grouping).
- Features available at prediction time (just before the closing quiz): `prior_score` (placement test), `session_minutes`, `hints_used` (both finished by the time the quiz starts), `enrol_year`, `school_code`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss
from sklearn.base import clone

df = pd.read_csv('data.csv')
print(df.shape, df.passed_quiz.mean())

## 1. Set aside the test set (whole students, 20%)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, df.passed_quiz, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert set(train.student_key).isdisjoint(test.student_key)
print(len(train), len(test), train.passed_quiz.mean(), test.passed_quiz.mean())
# test is not touched again until the final section
y_train, g_train = train.passed_quiz, train.student_key

## 2. Candidate models (named before scoring any)
1. **baseline** - predicts the training pass rate for every row.
2. **logreg_core** - logistic regression on `prior_score`, `session_minutes`, `hints_used` (scaled).
3. **logreg_all** - logistic regression on the core features plus `enrol_year` (numeric) and one-hot `school_code`.
4. **gbm_core** - histogram gradient boosting on the core features.
5. **gbm_all** - histogram gradient boosting on all features.

Selection metric: mean grouped-CV ROC AUC; ties within one SD go to the simplest model. The decision threshold is then chosen on the out-of-fold predictions of the winner by maximising balanced accuracy.

In [ ]:
core = ['prior_score', 'session_minutes', 'hints_used']
allf = core + ['enrol_year', 'school_code']

def lr(cols, cat=()):
    num = [c for c in cols if c not in cat]
    parts = [('n', StandardScaler(), num)]
    if cat:
        parts.append(('c', OneHotEncoder(handle_unknown='ignore'), list(cat)))
    return Pipeline([('pre', ColumnTransformer(parts)), ('m', LogisticRegression(max_iter=1000))])

def gbm():
    return Pipeline([('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))])

candidates = {
    'baseline':    (Pipeline([('m', DummyClassifier(strategy='prior'))]), core),
    'logreg_core': (lr(core), core),
    'logreg_all':  (lr(allf, cat=['school_code']), allf),
    'gbm_core':    (gbm(), core),
    'gbm_all':     (gbm(), allf),
}

## 3. Grouped cross-validation on the training rows only
Scaling and encoding live inside each pipeline, so they are fit within each training fold.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1)
folds = list(cv.split(train, y_train, groups=g_train))
oof, rows = {}, []
for name, (pipe, cols) in candidates.items():
    p = np.zeros(len(train)); aucs = []
    for a, b in folds:
        m = clone(pipe).fit(train.loc[a, cols], y_train[a])
        p[b] = m.predict_proba(train.loc[b, cols])[:, 1]
        aucs.append(roc_auc_score(y_train[b], p[b]))
    oof[name] = p
    rows.append((name, np.mean(aucs), np.std(aucs), brier_score_loss(y_train, p)))
cvres = pd.DataFrame(rows, columns=['model', 'cv_auc', 'auc_sd', 'brier']).sort_values('cv_auc', ascending=False)
cvres

## 4. Choose model and threshold on CV only

In [ ]:
order = ['logreg_core', 'logreg_all', 'gbm_core', 'gbm_all']
top = cvres.iloc[0]
ok = cvres[cvres.cv_auc >= top.cv_auc - top.auc_sd].model.tolist()
chosen = next(m for m in order if m in ok)
ths = np.linspace(0.1, 0.9, 81)
bal = [balanced_accuracy_score(y_train, oof[chosen] >= t) for t in ths]
thr = float(ths[int(np.argmax(bal))])
print('best by CV AUC:', top.model, '| chosen (simplest within 1 SD):', chosen,
      '| threshold:', round(thr, 3), '| CV balanced acc:', round(max(bal), 3))

**Recorded choice (fixed before the test set is scored):** the model in `chosen` above, with threshold `thr`, its listed features and the hyperparameters in its definition. No tuning happens after this point.

## 5. Refit on all training rows; score the test set once (headline)

In [ ]:
pipe, cols = candidates[chosen]
final = clone(pipe).fit(train[cols], y_train)
pt = final.predict_proba(test[cols])[:, 1]
print('TEST ROC AUC      :', round(roc_auc_score(test.passed_quiz, pt), 4))
print('TEST balanced acc :', round(balanced_accuracy_score(test.passed_quiz, pt >= thr), 4), f'(threshold {thr:.2f})')
print('TEST Brier        :', round(brier_score_loss(test.passed_quiz, pt), 4))

## Caveats
- No timestamp exists, so the result speaks to **new students**, not to future sessions of known students.
- `session_minutes` and `hints_used` are only complete once the session ends, so this is a pre-quiz predictor rather than a start-of-session one.
- Compare the test AUC against 0.5 (the baseline) and against the CV AUC of the chosen model; a large gap would signal optimism in selection.